**О проекте:**  
Данный проект выполнен в рамках практического обучения. Базовый пайплайн обработки данных (EDA, первичная очистка) разрабатывался под менторством преподавателя (я реализовывал код step-by-step в ходе урока).

**Моя самостоятельная часть работы (финальное задание) включала:**
* Обработку признака living_region
* Самостоятельную генерацию новых признаков
* Обучение модели машинного обучения на тренировочных данных.
* Подбор гиперпараметров
* Получите предсказания модели на тестовых данных test_data


##File download

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
pip install category_encoders

In [ ]:
! pip install wldhx.yadisk-direct
! curl -L $(yadisk-direct https://disk.yandex.com/d/sknuSa3xoNBsDw) -o bank-issues-data.zip
! unzip -qq bank-issues-data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 3473k  100 3473k    0     0   952k      0  0:00:03  0:00:03 --:--:-- 2091k
replace __MACOSX/._bank-issues-data? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/sampleSubmission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._sampleSubmission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [ ]:
! unzip -qq bank-issues-data.zip

replace __MACOSX/._bank-issues-data? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/sampleSubmission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._sampleSubmission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace bank-issues-data/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/bank-issues-data/._train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


##Data processing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_data = pd.read_csv('./bank-issues-data/train.csv')
test_data = pd.read_csv('./bank-issues-data/test.csv')

In [ ]:
train_data

,client_id,gender,age,marital_status,job_position,credit_sum,credit_month,tariff_id,score_shk,education,living_region,monthly_income,credit_count,overdue_credit_count,open_account_flg
0,1,M,25,UNM,SPC,26389.0,10,1.32,0.584105,SCH,ОБЛ КУРСКАЯ,35000.0,2.0,0.0,1
1,2,F,37,MAR,SPC,19588.0,12,1.43,0.718935,SCH,РЕСПУБЛИКА ТАТАРСТАН,15000.0,0.0,0.0,1
2,3,F,28,UNM,SPC,53669.0,18,1.10,0.586015,GRD,МОСКВА Г,70000.0,4.0,0.0,1
3,4,M,34,MAR,SPC,26349.0,10,1.43,0.655703,SCH,СВЕРДЛОВСКАЯ ОБЛАСТЬ,42500.0,4.0,0.0,0
4,5,F,43,MAR,UMN,11589.0,10,1.10,0.271893,GRD,РЯЗАНСКАЯ ОБЛАСТЬ,20000.0,3.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119513,119514,F,29,UNM,SPC,21490.0,12,1.40,0.551201,GRD,САНКТ-ПЕТЕРБУРГ Г,65000.0,2.0,0.0,0
119514,119515,M,40,MAR,SPC,45028.0,10,1.60,0.698376,SCH,ТАТАРСТАН РЕСП,50000.0,NaN,NaN,0
119515,119516,F,22,UNM,SPC,20990.0,12,1.90,0.570818,GRD,ОБЛ КУРГАНСКАЯ,20000.0,1.0,0.0,1
119516,119517,F,34,UNM,SPC,17108.0,10,1.10,0.442541,GRD,КРАСНОДАРСКИЙ КРАЙ,17000.0,0.0,0.0,0


Поля данных:
- **client_id** — Уникальный идентификатор клиента
- **gender** — Пол
- **age** — Возраст (в годах)
- **marital_status** — Семейное положение.
    Возможные значения:
    - UNM : Холост/не замужем
    - DIV : Резведен (а)
    - MAR : Женат/замужем
    - WID : Вдовец, вдова
    - CIV : Гражданский брак
- **job_position** — Работа.
    Возможные значения:
    - SPC : Неруководящий сотрудник - специалист
    - DIR : Руководитель организации
    - HSK : Домохозяйка
    - WOI : Работает на ИП
    - WRK : Неруководящий сотрудник - рабочий
    - ATP : Неруководящий сотрудник - обслуживающий персонал
    - WRP : Работающий пенсионер
    - UMN : Руководитель подразделения
    - NOR : Не работает
    - NS : Пенсионер
    - BIS : Собственный бизнес
    - INP : Индивидуальный предприниматель
- **credit_sum** — Сумма кредита
- **credit_month** — Срок кредитования в месяцах
- **tariff_id** — Номер предлагаемого тарифа
- **education** — Тип образования.
    Возможные знаяения:
    - SCH : Начальное, среднее
    - PGR : Второе высшее
    - GRD : Высшее
    - UGR : Неполное высшее
    - ACD : Ученая степень
- **living_region** — Регион проживания
- **monthly_income** — Зарплата в месяц
- **credit_count** — Количество кредитов у клиента
- **overdue_credit_count** — Количество просроченных кредитов клиента
- **open_account_flag** — Целевая переменная -- выберет клиент наш банк или нет

In [ ]:
y_train = train_data['open_account_flg']
X_train = train_data.drop(columns='open_account_flg')

train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119518 entries, 0 to 119517
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   client_id             119518 non-null  int64  
 1   gender                119518 non-null  object 
 2   age                   119518 non-null  int64  
 3   marital_status        119518 non-null  object 
 4   job_position          119518 non-null  object 
 5   credit_sum            119518 non-null  float64
 6   credit_month          119518 non-null  int64  
 7   tariff_id             119518 non-null  float64
 8   score_shk             119518 non-null  float64
 9   education             119518 non-null  object 
 10  living_region         119385 non-null  object 
 11  monthly_income        119518 non-null  float64
 12  credit_count          113032 non-null  float64
 13  overdue_credit_count  113032 non-null  float64
 14  open_account_flg      119518 non-null  int64  
dtype

In [ ]:
columns_to_drop = ['client_id']

train_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')
test_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')

display(train_data.head())

,gender,age,marital_status,job_position,credit_sum,credit_month,tariff_id,score_shk,education,living_region,monthly_income,credit_count,overdue_credit_count,open_account_flg
0,M,25,UNM,SPC,26389.0,10,1.32,0.584105,SCH,ОБЛ КУРСКАЯ,35000.0,2.0,0.0,1
1,F,37,MAR,SPC,19588.0,12,1.43,0.718935,SCH,РЕСПУБЛИКА ТАТАРСТАН,15000.0,0.0,0.0,1
2,F,28,UNM,SPC,53669.0,18,1.10,0.586015,GRD,МОСКВА Г,70000.0,4.0,0.0,1
3,M,34,MAR,SPC,26349.0,10,1.43,0.655703,SCH,СВЕРДЛОВСКАЯ ОБЛАСТЬ,42500.0,4.0,0.0,0
4,F,43,MAR,UMN,11589.0,10,1.10,0.271893,GRD,РЯЗАНСКАЯ ОБЛАСТЬ,20000.0,3.0,0.0,0


##Заполняю нулевые данные

In [ ]:
np.unique(train_data['credit_count'], return_counts=True)

(array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
        13., 14., 15., 16., 17., 18., 19., nan]),
 array([18219, 31714, 25799, 16707,  9826,  5285,  2760,  1345,   707,
          326,   141,    90,    56,    25,    12,    10,     2,     4,
            1,     3,  6486]))

In [ ]:
train_data['job_position'].dtype

dtype('O')

In [ ]:
#выбираем все столбцы с типом данных int или float, чтобы использовать ее в KNN inputer
num_col = [column for column in train_data.columns if train_data[column].dtype != 'O']
num_col

['age',
 'credit_sum',
 'credit_month',
 'tariff_id',
 'score_shk',
 'monthly_income',
 'credit_count',
 'overdue_credit_count',
 'open_account_flg']

In [ ]:
from sklearn.impute import KNNImputer

knn_inputer = KNNImputer(n_neighbors=2)
knn_inputer.fit(train_data[num_col])

transformed_overdue_credit_count = knn_inputer.transform(train_data[num_col])[:, 8]
train_data['overdue_credit_count'] = transformed_overdue_credit_count

In [ ]:
train_data['overdue_credit_count'].isna().any()

np.False_

##Обработка категориальных признаков

In [ ]:
categorical_columns = [column for column in train_data.columns if train_data[column].dtype == 'O']
categorical_columns += ['tariff_id'] #tariff_id также является категориальной переменной не смотря на числовой тип данных
categorical_columns = [x for x in categorical_columns if x != 'living_region']
np.array(categorical_columns)

array(['gender', 'marital_status', 'job_position', 'education',
       'tariff_id'], dtype='<U14')

###One hot encoder

In [ ]:
# использую именно этот метод, так как он не создаёт мнимых иерархий внутренних признакво,
# что делает его лучше чем label encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='if_binary', sparse_output=False)
ohe.fit(train_data[categorical_columns])

new_categorical_col = ohe.transform(train_data[categorical_columns])
new_categorical_col

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]])

In [ ]:
ohe.get_feature_names_out()

array(['gender_M', 'marital_status_CIV', 'marital_status_DIV',
       'marital_status_MAR', 'marital_status_UNM', 'marital_status_WID',
       'job_position_ATP', 'job_position_BIS', 'job_position_BIU',
       'job_position_DIR', 'job_position_HSK', 'job_position_INP',
       'job_position_NOR', 'job_position_PNA', 'job_position_PNI',
       'job_position_PNS', 'job_position_PNV', 'job_position_SPC',
       'job_position_UMN', 'job_position_WOI', 'job_position_WRK',
       'job_position_WRP', 'education_ACD', 'education_GRD',
       'education_PGR', 'education_SCH', 'education_UGR', 'tariff_id_1.0',
       'tariff_id_1.1', 'tariff_id_1.16', 'tariff_id_1.17',
       'tariff_id_1.18', 'tariff_id_1.19', 'tariff_id_1.2',
       'tariff_id_1.21', 'tariff_id_1.22', 'tariff_id_1.23',
       'tariff_id_1.24', 'tariff_id_1.25', 'tariff_id_1.26',
       'tariff_id_1.27', 'tariff_id_1.28', 'tariff_id_1.29',
       'tariff_id_1.3', 'tariff_id_1.32', 'tariff_id_1.4',
       'tariff_id_1.41', 'tarif

In [ ]:
new_train_columns = pd.DataFrame(new_categorical_col, columns=ohe.get_feature_names_out())
train_data = train_data.drop(columns=categorical_columns)
train_data = pd.concat([train_data, new_train_columns], axis=1)
train_data.head()

,age,credit_sum,credit_month,score_shk,living_region,monthly_income,credit_count,overdue_credit_count,open_account_flg,gender_M,...,tariff_id_1.48,tariff_id_1.5,tariff_id_1.52,tariff_id_1.56,tariff_id_1.6,tariff_id_1.7,tariff_id_1.9,tariff_id_1.91,tariff_id_1.94,tariff_id_1.96
0,25,26389.0,10,0.584105,ОБЛ КУРСКАЯ,35000.0,2.0,1.0,1,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,37,19588.0,12,0.718935,РЕСПУБЛИКА ТАТАРСТАН,15000.0,0.0,1.0,1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,28,53669.0,18,0.586015,МОСКВА Г,70000.0,4.0,1.0,1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,34,26349.0,10,0.655703,СВЕРДЛОВСКАЯ ОБЛАСТЬ,42500.0,4.0,0.0,0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,43,11589.0,10,0.271893,РЯЗАНСКАЯ ОБЛАСТЬ,20000.0,3.0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# дополнительно обрабатываем living_region:
# для этого: города разделим по названию (убрав вариации г; г.; город; и тд.)
# те города которые встречаются меньше 10 раз мы удаляем и не считываем (тем самым убираем выборсы)
# если в тест данных встречаются регионы которых нет в трейн данных мы их удаляем

import re

In [ ]:
def clean_region(text):
    if pd.isna(text):
        return 'unknown'
    text = text.lower().strip()
    text = re.sub(r'\b(г|гор|город|п|пос|село|деревня|д)\.?\s+', '', text)
    return text

train_data['living_region'] = train_data['living_region'].apply(clean_region)
test_data['living_region'] = test_data['living_region'].apply(clean_region)

region_counts = train_data['living_region'].value_counts()
valid_regions = region_counts[region_counts >= 10].index

train_data = train_data[train_data['living_region'].isin(valid_regions)].reset_index(drop=True)

test_data = test_data[test_data['living_region'].isin(valid_regions)].reset_index(drop=True)

print(f"Уникальных регионов после обработки: {len(train_data['living_region'].unique())}")
display(train_data['living_region'].value_counts().head())

Уникальных регионов после обработки: 202


,count
living_region,
обл московская,8564
краснодарский край,5176
санкт-петербург,3991
москва,3881
татарстан респ,3635


In [ ]:
import category_encoders as ce

target_encoder = ce.TargetEncoder(cols=['living_region'], smoothing=10)

# Исправлено название колонки с 'open_account_flag' на 'open_account_flg'
train_data['living_region_encoded'] = target_encoder.fit_transform(
    train_data['living_region'],
    train_data['open_account_flg']
)

test_data['living_region_encoded'] = target_encoder.transform(test_data['living_region'])

train_data = train_data.drop('living_region', axis=1)
test_data = test_data.drop('living_region', axis=1)

train_data['living_region_encoded']

,living_region_encoded
0,0.175525
1,0.104428
2,0.171383
3,0.158156
4,0.117949
...,...
119256,0.195070
119257,0.160660
119258,0.156863
119259,0.205757


##Обучение модели на тренировочных данных

In [ ]:
print("Есть ли таргет в train_data?", 'open_account_flg' in train_data.columns)
print("Есть ли таргет в test_data?", 'open_account_flg' in test_data.columns)

Есть ли таргет в train_data? True
Есть ли таргет в test_data? False


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# подготовка train
X = pd.get_dummies(train_data.drop(['open_account_flg', 'overdue_credit_count'], axis=1))
y = train_data['open_account_flg']

# обучение модели
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X, y)
print("Модель успешно обучена!")

# подготовка test
if 'id' in test_data.columns:
    ids = test_data['id']

X_test_final = pd.get_dummies(
    test_data.drop(columns=['open_account_flg', 'overdue_credit_count'], errors='ignore')
)
# выравниваем признаки
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)


Модель успешно обучена!


In [ ]:
y_pred_proba = rf_model.predict_proba(X_test_final)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_train = rf_model.predict_proba(X)[:,1]

roc_auc = roc_auc_score(y, y_pred_train)
print("ROC-AUC:", roc_auc)

ROC-AUC: 0.9999798408965767


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = pd.get_dummies(train_data.drop('open_account_flg', axis=1))
y = train_data['open_account_flg']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_model.fit(X_train, y_train)

y_pred_val = rf_model.predict_proba(X_val)[:,1]

roc_auc = roc_auc_score(y_val, y_pred_val)
print("Validation ROC-AUC:", roc_auc)


Validation ROC-AUC: 1.0


In [ ]:
corr = train_data.corr(numeric_only=True)['open_account_flg'].sort_values(ascending=False)
print(corr.head(15))


overdue_credit_count     1.000000
open_account_flg         1.000000
tariff_id_1.32           0.167666
living_region_encoded    0.121541
education_SCH            0.079373
job_position_PNA         0.079258
marital_status_UNM       0.062608
score_shk                0.049319
gender_M                 0.043414
tariff_id_1.3            0.042804
job_position_ATP         0.037995
credit_count             0.037318
tariff_id_1.1            0.031636
tariff_id_1.18           0.026752
credit_month             0.026355
Name: open_account_flg, dtype: float64
